# NB02: London Air Quality Data Transformation

**DS105W Mini-Project 1, Data for Data Science (Winter Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Student Notebook**
- 📅 Date: 23 February 2026
- 👤 Name: Joshua Andrew
- 📛 Candidate Number: 67321
- 🎯 Purpose: Transform raw JSON air pollution data into a clean CSV for analysis.
- ❓  **Question: Does London's air clean up on weekends?**

</div>

## Overview

This notebook takes the JSON file saved in NB01 and turns it into a CSV that NB03 can use for analysis. The JSON from the API has a nested structure (dicts within dicts), so the main job here is flattening that into a proper table, converting the UNIX timestamps to dates, and tagging each hour as weekday or weekend so that NB03 can do the comparison directly.

## Section 1: Setup

⚙️ **Importing libraries**

In [1]:
import json

import numpy as np
import pandas as pd

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## Section 2: Load the JSON Data

Loading the file saved in NB01. This is the same `json.load()` pattern from 🖥️ [W03 Lecture](https://lse-dsi.github.io/DS105/2025-2026/winter-term/weeks/week03/lecture.html).

In [2]:
with open("../data/london_air_pollution_2020_2025.json", "r") as f:
    raw_data = json.load(f)

print(f"Keys: {list(raw_data.keys())}")
print(f"Number of records: {len(raw_data['list'])}")

Keys: ['coord', 'list']
Number of records: 44064


In [3]:
# Remind myself what one record looks like
raw_data["list"][0]

{'main': {'aqi': 2},
 'components': {'co': 347.14,
  'no': 33.53,
  'no2': 41.13,
  'o3': 0.01,
  'so2': 7.51,
  'pm2_5': 18.81,
  'pm10': 21.35,
  'nh3': 0.25},
 'dt': 1606435200}

## Section 3: Parse into a DataFrame

Each record has `"dt"` (timestamp), `"main"` (a dict containing the AQI), and `"components"` (a dict with pollutant values). I need to flatten this into a single table with one column per field.

In [4]:
# Start by passing the list of records to pd.DataFrame
df = pd.DataFrame(raw_data["list"])

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (44064, 3)
Columns: ['main', 'components', 'dt']


,main,components,dt
0,{'aqi': 2},"{'co': 347.14, 'no': 33.53, 'no2': 41.13, 'o3'...",1606435200
1,{'aqi': 2},"{'co': 293.73, 'no': 11.18, 'no2': 42.16, 'o3'...",1606438800
2,{'aqi': 2},"{'co': 277.04, 'no': 5.64, 'no2': 41.81, 'o3':...",1606442400
3,{'aqi': 2},"{'co': 277.04, 'no': 4.75, 'no2': 41.13, 'o3':...",1606446000
4,{'aqi': 2},"{'co': 277.04, 'no': 4.47, 'no2': 40.44, 'o3':...",1606449600


💭 **Personal Reflection Notes:**

When using `pd.DataFrame(raw_data["list"])`, it gave me three columns: `dt`, `main`, and `components`. However, the `main` and `components` columns still contain dicts, not simple values. In the W05 Lecture, we created DataFrames from flat dictionaries like `pd.DataFrame({'date': dates, 'temp': temps})` where each value was already a plain list. My data is different because the values are nested.

Thus, to extract the actual numbers, I can use `.apply()` with a lambda function. This was taught in the W05 Lecture where we used `.apply()` to apply a function to every row. In this case, I'm applying it to a column of dicts to pull out a specific key from each one.

In [5]:
# Extract AQI from the nested "main" dict using .apply() (W05 Lecture Section 1)
df["aqi"] = df["main"].apply(lambda x: x["aqi"])

# Extract each pollutant from the "components" dict
df["co"] = df["components"].apply(lambda x: x["co"])
df["no"] = df["components"].apply(lambda x: x["no"])
df["no2"] = df["components"].apply(lambda x: x["no2"])
df["o3"] = df["components"].apply(lambda x: x["o3"])
df["so2"] = df["components"].apply(lambda x: x["so2"])
df["pm2_5"] = df["components"].apply(lambda x: x["pm2_5"])
df["pm10"] = df["components"].apply(lambda x: x["pm10"])
df["nh3"] = df["components"].apply(lambda x: x["nh3"])

In [6]:
# Keep only the columns I need, dropping the original nested ones
df = df[["dt", "aqi", "co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]]

print(f"Columns: {df.columns.tolist()}")
df.head()

Columns: ['dt', 'aqi', 'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']


,dt,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,1606435200,2.0,347.14,33.53,41.13,0.01,7.51,18.81,21.35,0.25
1,1606438800,2.0,293.73,11.18,42.16,0.21,7.27,15.68,18.17,0.01
2,1606442400,2.0,277.04,5.64,41.81,0.32,7.33,15.31,17.65,0.01
3,1606446000,2.0,277.04,4.75,41.13,0.40,7.57,15.78,18.02,0.02
4,1606449600,2.0,277.04,4.47,40.44,0.43,7.87,16.73,18.96,0.03


## Section 4: DateTime Conversion

The `dt` column contains UNIX timestamps (seconds since 1 Jan 1970). I need to convert them to datetime objects so I can extract the day of the week. The 🖥️ [W05 Lecture](https://moodle.lse.ac.uk/mod/page/view.php?id=1598012) Section 2 showed how to use `pd.to_datetime()` on date strings and the `.dt` accessor to pull out components. My situation is slightly different since I have UNIX integers rather than date strings, so I looked up the [pandas documentation for pd.to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) and found the `unit="s"` parameter which tells pandas the input is in seconds.

In [7]:
# Convert UNIX timestamp to datetime
df["datetime"] = pd.to_datetime(df["dt"], unit="s")

df[["dt", "datetime"]].head()

,dt,datetime
0,1606435200,2020-11-27 00:00:00
1,1606438800,2020-11-27 01:00:00
2,1606442400,2020-11-27 02:00:00
3,1606446000,2020-11-27 03:00:00
4,1606449600,2020-11-27 04:00:00


In [8]:
print(f"Original dt column type: {df['dt'].dtype}")
print(f"New datetime type: {df['datetime'].dtype}")

Original dt column type: int64
New datetime type: datetime64[s]


💭 **Personal Reflection Notes:**

The W05 Lecture used `pd.to_datetime()` on string dates like `"2006-01-01"`. My timestamps are integers (UNIX seconds), so I needed the `unit="s"` parameter to tell pandas what the numbers represent. I found this in the [pandas to_datetime docs](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html). I confirmed it worked by checking that the first timestamp converts to late November 2020 and the dtype changed from `int64` to `datetime64`.

## Section 5: Extract Date Components

Using the `.dt` accessor to pull out the parts I need for the weekday vs weekend analysis. The W05 Lecture slides showed `.dt.year`, `.dt.month`, `.dt.day`, and `.dt.dayofweek`. I'm also using `.dt.hour` and `.dt.date` which were not in the slides but follow the exact same pattern. I checked the [pandas .dt accessor docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.html) to confirm these exist.

The `.dt.dayofweek` values follow the pandas convention: Monday = 0, Tuesday = 1, Wednesday = 2, Thursday = 3, Friday = 4, Saturday = 5, Sunday = 6.

In [9]:
# Extract date components using .dt accessor (W05 Lecture Section 2)
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek  # Monday=0, Sunday=6
df["month"] = df["datetime"].dt.month

df[["datetime", "date", "hour", "dayofweek", "month"]].head(10)

,datetime,date,hour,dayofweek,month
0,2020-11-27 00:00:00,2020-11-27,0,4,11
1,2020-11-27 01:00:00,2020-11-27,1,4,11
2,2020-11-27 02:00:00,2020-11-27,2,4,11
3,2020-11-27 03:00:00,2020-11-27,3,4,11
4,2020-11-27 04:00:00,2020-11-27,4,4,11
5,2020-11-27 05:00:00,2020-11-27,5,4,11
6,2020-11-27 06:00:00,2020-11-27,6,4,11
7,2020-11-27 07:00:00,2020-11-27,7,4,11
8,2020-11-27 08:00:00,2020-11-27,8,4,11
9,2020-11-27 09:00:00,2020-11-27,9,4,11


## Section 6: Tag Weekday vs Weekend

This is the key column for the research question. Since Saturday = 5 and Sunday = 6, any `dayofweek >= 5` is a weekend day. Using this, I can create a boolean column using a vectorised comparison, the same approach as `weather_df['is_hot'] = weather_df['temp'] >= 28` from the W04 Lecture.

In [10]:
# Vectorised boolean: True if Saturday (5) or Sunday (6)
df["is_weekend"] = df["dayofweek"] >= 5

df[["datetime", "dayofweek", "is_weekend"]].head(10)

,datetime,dayofweek,is_weekend
0,2020-11-27 00:00:00,4,False
1,2020-11-27 01:00:00,4,False
2,2020-11-27 02:00:00,4,False
3,2020-11-27 03:00:00,4,False
4,2020-11-27 04:00:00,4,False
5,2020-11-27 05:00:00,4,False
6,2020-11-27 06:00:00,4,False
7,2020-11-27 07:00:00,4,False
8,2020-11-27 08:00:00,4,False
9,2020-11-27 09:00:00,4,False


In [11]:
# Also create a readable string version using np.where (W04 Lecture)
df["day_type"] = np.where(df["is_weekend"], "Weekend", "Weekday")

# Quick check: how many weekday vs weekend hours?
print(df["day_type"].value_counts())

day_type
Weekday    31486
Weekend    12578
Name: count, dtype: int64


💭 **Personal Reflection Notes:**

**Weekend definition decision:** I defined weekend as Saturday and Sunday only (dayofweek 5 and 6). The project brief mentioned considering whether Friday evening should count, or whether public holidays should be included. I decided to keep it simple with the standard Saturday/Sunday definition for a few reasons. Firstly, it is the most common understanding of "weekend". Secondly, its easy to implement with a single vectorised comparison (`>= 5`). Lastly, adding public holidays would require additional steps that could form an interesting extension, but as an overview just seeing the difference between weekdays and weekends should be a good starting point. If my NB03 analysis shows interesting patterns, I could revisit this.

**Temporal aggregation decision:** The brief also asks about temporal aggregation (hourly averages, daily means, etc.). I decided to keep the data at hourly granularity in this CSV rather than aggregating here. This gives NB03 more flexibility. For example, I could look at how pollution changes throughout the day on weekdays vs weekends (hourly patterns), or I could aggregate to daily means first if I want a simpler comparison. Doing the aggregation in NB03 means I can try different approaches without having to come back and re-run NB02.

## Section 7: Check for Data Gaps

Since the data spans about 5 years, there might be hours where the API has no readings. Now that I have the weekend tagging, I can also check whether the gaps are spread evenly.

In [12]:
# Tag each row as having a gap before it
# .diff() calculates the time difference between each row and the previous one
# Then np.where checks if that difference is exactly 1 hour or not
df["has_gap"] = np.where(df["datetime"].diff() == pd.Timedelta(hours=1), False, True)

# The first row always shows as a gap because .diff() has no previous row
# So the real gap count is .sum() minus 1
print(f"Total records: {len(df)}")
print(f"Records with a gap before them: {df['has_gap'].sum() - 1}")

Total records: 44064
Records with a gap before them: 16


In [13]:
# Check gap distribution across weekdays and weekends
# Filter to just the gap rows using boolean filtering (W05 Lecture: df[df['temp'] > 25])
gap_rows = df[df["has_gap"] == True]
print("Gaps by day type:")
print(gap_rows["day_type"].value_counts())
print("(includes 1 false positive from the first row of .diff())")

Gaps by day type:
day_type
Weekday    12
Weekend     5
Name: count, dtype: int64
(includes 1 false positive from the first row of .diff())


💭 **Personal Reflection Notes:**

I wanted to check whether the data gaps are spread across both weekdays and weekends, since clustered gaps could bias the NB03 analysis. To do this I needed to compare consecutive timestamps, which is not something we covered in the course. The course taught vectorised operations on single columns (like `df['temp'] >= 28`), but not operations that compare each row to the one before it.

I found `.diff()` and `pd.Timedelta` in the [pandas .diff() documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.diff.html). `.diff()` calculates the difference between each row and the previous one, and `pd.Timedelta(hours=1)` creates a 1-hour time value to compare against. I tested `.diff()` on a small slice first (`df['datetime'].head(10).diff()`) to make sure the output was what I expected before running it on the full dataset. I then combined it with `np.where()` (W04 Lecture) to create a boolean column, and used boolean filtering (`df[df['has_gap'] == True]`, same pattern as `df[df['temperature'] > 25]` from the W05 Lecture slides) with `.value_counts()` (W04 Lab) to check the gap distribution. The gaps appear on both weekdays and weekends, so they should not bias the analysis.

## Section 8: Clean Outlier Values

Before saving, I want to check the data for any obviously wrong values.

In [14]:
# Summary statistics for the pollutant columns
pollutant_cols = ["co", "no", "no2", "o3", "so2", "pm2_5", "pm10", "nh3"]
df[pollutant_cols].describe()

,co,no,no2,o3,so2,pm2_5,pm10,nh3
count,44064.000000,44064.000000,44064.000000,44064.000000,44064.000000,44064.000000,44064.000000,44064.000000
mean,239.485032,6.764095,18.858563,51.365479,7.704103,6.806871,8.430080,0.908936
std,94.309418,24.869802,69.610211,56.538909,7.676688,9.104027,48.671835,1.084641
min,81.200000,0.000000,-9999.000000,-9999.000000,0.550000,0.000000,-9999.000000,0.000000
25%,190.260000,0.000000,7.540000,30.760000,3.670000,1.910000,2.960000,0.320000
50%,230.310000,0.290000,13.540000,55.070000,5.720000,3.580000,5.490000,0.590000
75%,267.030000,1.600000,25.700000,72.240000,9.082500,7.810000,10.480000,1.070000
max,1535.420000,529.290000,174.110000,217.440000,188.830000,115.890000,117.470000,16.720000


The `.describe()` output shows some suspicious minimum values. Pollutant concentrations cannot be negative, so any negative values are likely error codes from the API rather than real readings.

In [15]:
# Count negative values per column
# Same pattern as is_hot_vectorised.sum() from W04 Lecture:
# boolean comparison creates True/False, .sum() counts the Trues
print("Negative values per column:")
print((df[pollutant_cols] < 0).sum())

Negative values per column:
co       0
no       0
no2      2
o3       1
so2      0
pm2_5    0
pm10     1
nh3      0
dtype: int64


In [16]:
# Replace negative values with NaN using np.where (W04 Lecture)
# np.nan is NumPy's missing value marker
# Keep the value if >= 0, otherwise mark as missing
df["co"] = np.where(df["co"] >= 0, df["co"], np.nan)
df["no"] = np.where(df["no"] >= 0, df["no"], np.nan)
df["no2"] = np.where(df["no2"] >= 0, df["no2"], np.nan)
df["o3"] = np.where(df["o3"] >= 0, df["o3"], np.nan)
df["so2"] = np.where(df["so2"] >= 0, df["so2"], np.nan)
df["pm2_5"] = np.where(df["pm2_5"] >= 0, df["pm2_5"], np.nan)
df["pm10"] = np.where(df["pm10"] >= 0, df["pm10"], np.nan)
df["nh3"] = np.where(df["nh3"] >= 0, df["nh3"], np.nan)

In [17]:
# Confirm the fix - minimums should now be >= 0
# Also check the 'count' row: if count < total rows, there are NaN values
print("After cleaning:")
df[pollutant_cols].describe()

After cleaning:


,co,no,no2,o3,so2,pm2_5,pm10,nh3
count,44064.000000,44064.000000,44062.000000,44063.000000,44064.000000,44064.000000,44063.000000,44064.000000
mean,239.485032,6.764095,19.313279,51.593570,7.704103,6.806871,8.657197,0.908936
std,94.309418,24.869802,17.035433,30.070263,7.676688,9.104027,9.800334,1.084641
min,81.200000,0.000000,0.690000,0.000000,0.550000,0.000000,0.000000,0.000000
25%,190.260000,0.000000,7.540000,30.760000,3.670000,1.910000,2.960000,0.320000
50%,230.310000,0.290000,13.540000,55.070000,5.720000,3.580000,5.490000,0.590000
75%,267.030000,1.600000,25.700000,72.240000,9.082500,7.810000,10.480000,1.070000
max,1535.420000,529.290000,174.110000,217.440000,188.830000,115.890000,117.470000,16.720000


💭 **Personal Reflection Notes:**

I found negative values in some pollutant columns, which are physically impossible since concentrations cannot be below zero. These are most likely error codes or missing data markers from the API. I chose to replace them with `NaN` rather than deleting the entire row, because a single bad reading in one pollutant should not throw away the other seven valid pollutant values for that hour.

`np.nan` is NumPy's way of representing a missing value. It was not shown in lectures, but I needed it as the replacement value inside `np.where()` (which was taught in the W04 Lecture). The course only showed `np.where()` with string outputs like `"Hot & Dry"`, but here I needed a missing value instead. I found `np.nan` in the [NumPy documentation](https://numpy.org/doc/stable/reference/constants.html#numpy.nan) and tested it by checking that `np.where(True, 5, np.nan)` returns `5` and `np.where(False, 5, np.nan)` returns `nan`. Pandas automatically skips `NaN` values in calculations like `.mean()` and `.groupby().mean()`, which I confirmed by checking the 'count' row in `.describe()` after cleaning. If the count is lower than the total rows, it means some values are now `NaN`.

I also noticed from `.describe()` that the pollutant scales are very different (CO in the hundreds, SO2 under 10), so I will need to keep this in mind during NB-03's analysis.

## Section 9: Verify and Save

Final checks before saving to CSV.

In [18]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Data types:")
print(df.dtypes)

Shape: (44064, 18)
Columns: ['dt', 'aqi', 'co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3', 'datetime', 'date', 'hour', 'dayofweek', 'month', 'is_weekend', 'day_type', 'has_gap']
Data types:
dt                    int64
aqi                 float64
co                  float64
no                  float64
no2                 float64
o3                  float64
so2                 float64
pm2_5               float64
pm10                float64
nh3                 float64
datetime      datetime64[s]
date                 object
hour                  int32
dayofweek             int32
month                 int32
is_weekend             bool
day_type                str
has_gap                bool
dtype: object


In [19]:
df.head()

,dt,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3,datetime,date,hour,dayofweek,month,is_weekend,day_type,has_gap
0,1606435200,2.0,347.14,33.53,41.13,0.01,7.51,18.81,21.35,0.25,2020-11-27 00:00:00,2020-11-27,0,4,11,False,Weekday,True
1,1606438800,2.0,293.73,11.18,42.16,0.21,7.27,15.68,18.17,0.01,2020-11-27 01:00:00,2020-11-27,1,4,11,False,Weekday,False
2,1606442400,2.0,277.04,5.64,41.81,0.32,7.33,15.31,17.65,0.01,2020-11-27 02:00:00,2020-11-27,2,4,11,False,Weekday,False
3,1606446000,2.0,277.04,4.75,41.13,0.40,7.57,15.78,18.02,0.02,2020-11-27 03:00:00,2020-11-27,3,4,11,False,Weekday,False
4,1606449600,2.0,277.04,4.47,40.44,0.43,7.87,16.73,18.96,0.03,2020-11-27 04:00:00,2020-11-27,4,4,11,False,Weekday,False


In [20]:
# Select the columns to save (excluding raw dt, datetime, and has_gap helper)
df_clean = df[["date", "hour", "dayofweek", "is_weekend", "day_type",
               "month", "aqi", "co", "no", "no2", "o3", "so2",
               "pm2_5", "pm10", "nh3"]]

print(f"Saving {df_clean.shape[0]} rows and {df_clean.shape[1]} columns")
df_clean.head()

Saving 44064 rows and 15 columns


,date,hour,dayofweek,is_weekend,day_type,month,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2020-11-27,0,4,False,Weekday,11,2.0,347.14,33.53,41.13,0.01,7.51,18.81,21.35,0.25
1,2020-11-27,1,4,False,Weekday,11,2.0,293.73,11.18,42.16,0.21,7.27,15.68,18.17,0.01
2,2020-11-27,2,4,False,Weekday,11,2.0,277.04,5.64,41.81,0.32,7.33,15.31,17.65,0.01
3,2020-11-27,3,4,False,Weekday,11,2.0,277.04,4.75,41.13,0.40,7.57,15.78,18.02,0.02
4,2020-11-27,4,4,False,Weekday,11,2.0,277.04,4.47,40.44,0.43,7.87,16.73,18.96,0.03


In [21]:
df_clean.to_csv("../data/london_air_quality_2020_2025.csv", index=False)

print("✅ Saved to data/london_air_quality_2020_2025.csv")

✅ Saved to data/london_air_quality_2020_2025.csv


In [22]:
# Quick verification: read it back to make sure it saved correctly
check_df = pd.read_csv("../data/london_air_quality_2020_2025.csv")
print(f"Read back: {check_df.shape}")
check_df.head()

Read back: (44064, 15)


,date,hour,dayofweek,is_weekend,day_type,month,aqi,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2020-11-27,0,4,False,Weekday,11,2.0,347.14,33.53,41.13,0.01,7.51,18.81,21.35,0.25
1,2020-11-27,1,4,False,Weekday,11,2.0,293.73,11.18,42.16,0.21,7.27,15.68,18.17,0.01
2,2020-11-27,2,4,False,Weekday,11,2.0,277.04,5.64,41.81,0.32,7.33,15.31,17.65,0.01
3,2020-11-27,3,4,False,Weekday,11,2.0,277.04,4.75,41.13,0.40,7.57,15.78,18.02,0.02
4,2020-11-27,4,4,False,Weekday,11,2.0,277.04,4.47,40.44,0.43,7.87,16.73,18.96,0.03


💭 **Personal Reflection Notes:**

I read the CSV back using `pd.read_csv()` to confirm the shape matched what I had before saving. This is the same verification approach from the W04 Practice, when the data was checked after saving. I used `index=False` in `to_csv()` because the default pandas row index would add an extra column that has no meaning for the analysis. One thing I noticed when reading back is that the `date` column comes back as a string rather than a date object, but that is fine since NB03 can convert it again if needed using `pd.to_datetime()`.

NB02 is done. The CSV in `data/london_air_quality_2020_2025.csv` has one row per hour with all pollutant concentrations, the AQI, and the weekday/weekend tagging. NB03 will read this file and produce two analytical insights.